In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import json
import pickle

with open('nslkdd_features.json', 'r') as f:
    features_meta = json.load(f)

column_names = [feat['name'] for feat in features_meta]

train_df = pd.read_csv('NSL-KDD/KDDTrain+.txt', header=None, names=column_names)
test_df = pd.read_csv('NSL-KDD/KDDTest+.txt', header=None, names=column_names)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

#Binary labels: normal=0, attack=1
train_df['binary_label'] = (train_df['label'] != 'normal').astype(int)
test_df['binary_label'] = (test_df['label'] != 'normal').astype(int)

print(f"\nTrain label distribution:\n{train_df['binary_label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['binary_label'].value_counts()}")

#Keep original labels for later
train_original_labels = train_df['label'].copy()
test_original_labels = test_df['label'].copy()

#Drop label - it is the target variable
#Drop difficulty - it is a metadata, which shows how hard is to detect the sample. It is not a real feature
train_df = train_df.drop(columns=['label', 'difficulty'])
test_df = test_df.drop(columns=['label', 'difficulty'])

#One-hot encoding for the following 3 columns
categorical_cols = ['protocol_type', 'service', 'flag']

#Combine train and test to get consistent one-hot encoding
combined = pd.concat([train_df[categorical_cols], test_df[categorical_cols]], axis=0)
combined_onehot = pd.get_dummies(combined, columns=categorical_cols)

#Split back the combined data into train and test
train_onehot = combined_onehot.iloc[:len(train_df)].reset_index(drop=True)
test_onehot = combined_onehot.iloc[len(train_df):].reset_index(drop=True)

#Replace categorical columns with one-hot encoded columns
train_numeric = train_df.drop(columns=categorical_cols).reset_index(drop=True)
test_numeric = test_df.drop(columns=categorical_cols).reset_index(drop=True)

#Separate binary_label before merging
y_train = train_numeric['binary_label'].values
y_test = test_numeric['binary_label'].values
train_numeric = train_numeric.drop(columns=['binary_label'])
test_numeric = test_numeric.drop(columns=['binary_label'])

#Combine numeric and one-hot encoded features
X_train_df = pd.concat([train_numeric, train_onehot], axis=1)
X_test_df = pd.concat([test_numeric, test_onehot], axis=1)

print(f"\nFeature columns after one-hot encoding: {X_train_df.shape[1]}")
print(f"X_train shape: {X_train_df.shape}")
print(f"X_test shape:  {X_test_df.shape}")

#Standardization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.values.astype(np.float64))
X_test = scaler.transform(X_test_df.values.astype(np.float64))

print(f"\nAfter standardization:")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_test shape:  {X_test.shape}, dtype: {X_test.dtype}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
print(f"X_train mean: {X_train.mean():.6f}")
print(f"X_train std: {X_train.std():.6f}")

#Saving preprocessed data, scaler and feature names for later
np.save('results/X_train.npy', X_train)
np.save('results/X_test.npy', X_test)
np.save('results/y_train.npy', y_train)
np.save('results/y_test.npy', y_test)

with open('results/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('results/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X_train_df.columns), f)

#Save original labels for later
np.save('results/train_original_labels.npy', train_original_labels.values)
np.save('results/test_original_labels.npy', test_original_labels.values)

print("\nPreprocessing complete! Files saved.")
print(f"Feature names: {list(X_train_df.columns)}")


Train shape: (125973, 43)
Test shape:  (22544, 43)

Train label distribution:
binary_label
0    67343
1    58630
Name: count, dtype: int64

Test label distribution:
binary_label
1    12833
0     9711
Name: count, dtype: int64

Feature columns after one-hot encoding: 122
X_train shape: (125973, 122)
X_test shape:  (22544, 122)

After standardization:
X_train shape: (125973, 122), dtype: float64
X_test shape:  (22544, 122), dtype: float64
y_train shape: (125973,), y_test shape: (22544,)
X_train mean (should be ~0): -0.000000
X_train std  (should be ~1): 0.995893

Preprocessing complete! Files saved.
Feature names (first 10): ['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerr